# 04 · E1: weather, collision frequency and severity (RQ1)

**RQ1:** After controlling for hour, day of week, season, year and the COVID-19 period, how are rain, snow, low visibility and low temperature associated with the number of collisions per hour and with the probability that a collision involves an injury or fatality?

* **Frequency:** negative binomial regression on 105,192 city-hours, checked against Poisson. Output: incidence rate ratios (IRR) with 95% CI. Baselines: the same model with time variables only, and the same-hour-of-week average.
* **Severity:** logistic regression (odds ratios) and gradient-boosted trees. The logistic model is also implemented in PyTorch. Trained on 2014–2022, tuned on 2023, tested on 2024–2025, with class weights. Primary metric: PR-AUC.

Reference categories: dry hour, visibility ≥ 5 km, temperature 10–20 °C.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src import models as M
ch = pd.read_parquet(PROC / "city_hour.parquet")
ch = ch[ch["temp_bin"] != "nan"].copy()
print(len(ch), "city-hours; mean", round(ch["collisions"].mean(), 2), "collisions/hour; variance", round(ch["collisions"].var(), 2))

## 1. Frequency: Poisson vs negative binomial

In [ ]:
fit = M.fit_count_models(ch, "collisions")
diag = pd.DataFrame({
    "value": [fit["pearson_dispersion"], fit["alpha"], fit["aic"]["poisson"], fit["aic"]["nb"], fit["aic"]["nb_time"],
              fit["lr_nb_vs_poisson"]["stat"], fit["lr_nb_vs_poisson"]["p"], fit["lr_weather"]["stat"],
              fit["lr_weather"]["df"], fit["lr_weather"]["p"]]},
    index=["Poisson Pearson dispersion", "NB alpha", "AIC Poisson", "AIC NB", "AIC NB time-only",
           "LR NB vs Poisson", "p", "LR weather terms (NB vs NB time-only)", "df", "p "])
diag.to_csv(TAB / "e1_frequency_diagnostics.csv")
diag.round(4)

In [ ]:
irr = M.ratio_table(fit["nb"])
irr.to_csv(TAB / "e1_frequency_irr.csv")
irr.round(3)

In [ ]:
labels = {"rain": "rain", "snow": "snow", "vis_lt1": "visibility < 1 km", "vis_1to5": "visibility 1-5 km",
          "snow_on_ground": "snow on ground", "np.log1p(hours_since_precip)": "log(1 + hours since precip.)",
          "C(temp_bin, Treatment('10_20'))[T.-50_-10]": "temp < -10 C", "C(temp_bin, Treatment('10_20'))[T.-10_0]": "temp -10 to 0 C",
          "C(temp_bin, Treatment('10_20'))[T.0_10]": "temp 0 to 10 C", "C(temp_bin, Treatment('10_20'))[T.20_50]": "temp > 20 C"}
def forest(tab, title, fname, xlabel):
    t = tab.loc[[k for k in labels if k in tab.index]].iloc[::-1]
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.errorbar(t["estimate"], range(len(t)), xerr=[t["estimate"] - t["ci_low"], t["ci_high"] - t["estimate"]],
                fmt="o", color="#4C72B0", capsize=3)
    ax.axvline(1, color="grey", lw=0.8, ls="--")
    ax.set_yticks(range(len(t))); ax.set_yticklabels([labels[i] for i in t.index])
    ax.set_xlabel(xlabel); ax.set_title(title)
    fig.savefig(FIG / fname); plt.show()
forest(irr, "Collision frequency: incidence rate ratios (95% CI)", "fig_e1_irr.png", "IRR (reference: dry, vis >= 5 km, 10-20 C)")

## 2. Frequency: out-of-sample comparison with baselines

Models are fitted on 2014–2023 and evaluated on 2024–2025. Year effects for the test years are set to the 2023 estimate, the last year observed. The same-hour-of-week baseline uses 2023 averages.

In [ ]:
import statsmodels.api as sm, statsmodels.formula.api as smf
train, test = ch[ch["year"] <= 2023].copy(), ch[ch["year"] >= 2024].copy()
test_pred = test.assign(year=2023)
f_full = M.formula("collisions", M.WEATHER_TERMS + M.CALENDAR_TERMS)
f_time = M.formula("collisions", M.CALENDAR_TERMS)
nb_full = M.fit_nb(f_full, train)
nb_time = M.fit_nb(f_time, train)
oos = pd.DataFrame({
    "same-hour-of-week average (2023)": M.mae_rmse(test["collisions"], M.hour_of_week_baseline(train[train["year"] == 2023], test, "collisions")),
    "NB, time variables only": M.mae_rmse(test["collisions"], nb_time.predict(test_pred)),
    "NB, time + weather": M.mae_rmse(test["collisions"], nb_full.predict(test_pred)),
}).T
oos.to_csv(TAB / "e1_frequency_out_of_sample.csv")
oos.round(3)

## 3. Severity: odds ratios (logistic regression, all collisions 2014–2025)

In [ ]:
cl = pd.read_parquet(PROC / "collision_level.parquet")
cl = cl[cl["temp_bin"] != "nan"].copy()
cl["severe"] = cl["severe"].astype(int)
sev_terms = M.WEATHER_TERMS + M.CALENDAR_TERMS + ["automobile", "motorcycle", "passenger", "bicycle", "pedestrian"]
for c in ["automobile", "motorcycle", "passenger", "bicycle", "pedestrian"]:
    cl[c] = cl[c].astype(int)
logit = smf.logit(M.formula("severe", sev_terms), cl).fit(disp=0, maxiter=200)
logit_time = smf.logit(M.formula("severe", M.CALENDAR_TERMS + ["automobile", "motorcycle", "passenger", "bicycle", "pedestrian"]), cl).fit(disp=0, maxiter=200)
from scipy import stats
lr = 2 * (logit.llf - logit_time.llf); dfw = logit.df_model - logit_time.df_model
print(f"LR test of weather terms: {lr:.1f} on {dfw:.0f} df, p = {stats.chi2.sf(lr, dfw):.3g}")
orr = M.ratio_table(logit)
orr.to_csv(TAB / "e1_severity_odds_ratios.csv")
orr.round(3)

In [ ]:
forest(orr, "Collision severity: odds ratios (95% CI)", "fig_e1_odds_ratios.png", "odds ratio of injury or fatality")

## 4. Severity: predictive models (train 2014–2022, tune 2023, test 2024–2025)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
sp = M.split_by_year(cl, cfg)
X = {k: M.severity_design(v) for k, v in sp.items()}
cols = X["train"].columns
X = {k: v.reindex(columns=cols, fill_value=0.0) for k, v in X.items()}
Xt = {k: M.severity_design(v, numeric=M.SEVERITY_TIME_ONLY + ["automobile", "motorcycle", "passenger", "bicycle", "pedestrian"],
                           categ=M.SEVERITY_TIME_CATEG) for k, v in sp.items()}
Xt = {k: v.reindex(columns=Xt["train"].columns, fill_value=0.0).drop(columns="log_hours_since_precip") for k, v in Xt.items()}
y = {k: v["severe"].to_numpy() for k, v in sp.items()}
print({k: (len(v), round(v.mean(), 3)) for k, v in y.items()})

In [ ]:
preds, val_preds = {}, {}
# Baseline 1: majority class ("not severe" for every collision)
preds["majority class"] = np.zeros(len(y["test"])); val_preds["majority class"] = np.zeros(len(y["val"]))
# Baseline 2: logistic regression with time variables and road users only
lt = LogisticRegression(class_weight="balanced", max_iter=3000).fit(Xt["train"], y["train"])
preds["logistic, time only"] = lt.predict_proba(Xt["test"])[:, 1]; val_preds["logistic, time only"] = lt.predict_proba(Xt["val"])[:, 1]
# Logistic regression (scikit-learn)
lg = LogisticRegression(class_weight="balanced", max_iter=3000).fit(X["train"], y["train"])
preds["logistic"] = lg.predict_proba(X["test"])[:, 1]; val_preds["logistic"] = lg.predict_proba(X["val"])[:, 1]
# Gradient-boosted trees, tuned on 2023
grid = cfg["severity"]["gbt_grid"]; tuning = []
for lr_ in grid["learning_rate"]:
    for leaves in grid["max_leaf_nodes"]:
        g = HistGradientBoostingClassifier(class_weight="balanced", learning_rate=lr_, max_leaf_nodes=leaves,
                                           max_iter=grid["max_iter"][0], early_stopping=False, random_state=cfg["seed"])
        g.fit(X["train"], y["train"])
        tuning.append({"learning_rate": lr_, "max_leaf_nodes": leaves, "val_PR_AUC": average_precision_score(y["val"], g.predict_proba(X["val"])[:, 1]), "model": g})
tun = pd.DataFrame(tuning)
display(tun.drop(columns="model").round(4))
gbt = tun.loc[tun["val_PR_AUC"].idxmax(), "model"]
preds["gradient-boosted trees"] = gbt.predict_proba(X["test"])[:, 1]; val_preds["gradient-boosted trees"] = gbt.predict_proba(X["val"])[:, 1]

### Logistic regression in PyTorch
The same model as above: one linear layer, a sigmoid output and binary cross-entropy with `pos_weight` for the class imbalance, trained with Adam. Early stopping uses validation PR-AUC.

In [ ]:
torch_model, torch_predict, history = M.train_torch_logistic(X["train"], y["train"], X["val"], y["val"], cfg, seed=cfg["seed"])
preds["logistic (PyTorch)"] = torch_predict(X["test"]); val_preds["logistic (PyTorch)"] = torch_predict(X["val"])
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(history["epoch"], history["train_loss"]); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("weighted BCE loss"); ax[0].set_title("Training loss")
ax[1].plot(history["epoch"], history["val_pr_auc"], color="#DD8452"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("PR-AUC"); ax[1].set_title("Validation PR-AUC (2023)")
fig.savefig(FIG / "fig_e1_torch_training.png"); plt.show()

In [ ]:
rows = {}
for name, p in preds.items():
    th = 0.5 if name == "majority class" else M.best_f1_threshold(y["val"], val_preds[name])
    rows[name] = M.classification_metrics(y["test"], p, th)
res = pd.DataFrame(rows).T
res.loc["majority class", ["PR-AUC", "ROC-AUC"]] = [y["test"].mean(), 0.5]
res["accuracy"] = [np.mean((p >= r["threshold"]).astype(int) == y["test"]) for p, (_, r) in zip(preds.values(), res.iterrows())]
res.to_csv(TAB / "e1_severity_test_metrics.csv")
res.round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for name in ["logistic, time only", "logistic", "logistic (PyTorch)", "gradient-boosted trees"]:
    pr, rc, _ = precision_recall_curve(y["test"], preds[name])
    ax[0].plot(rc, pr, label=f"{name} (AP={res.loc[name, 'PR-AUC']:.3f})")
ax[0].axhline(y["test"].mean(), color="grey", ls=":", label=f"majority / no skill ({y['test'].mean():.3f})")
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("Precision-recall, test 2024-2025"); ax[0].legend(frameon=False, fontsize=7)
for name in ["logistic", "gradient-boosted trees"]:
    fp, mp = calibration_curve(y["test"], preds[name], n_bins=10, strategy="quantile")
    ax[1].plot(mp, fp, marker="o", label=name)
ax[1].plot([0, 1], [0, 1], color="grey", ls=":")
ax[1].set_xlabel("mean predicted probability"); ax[1].set_ylabel("observed share severe"); ax[1].set_title("Calibration (class-weighted models)"); ax[1].legend(frameon=False, fontsize=7)
fig.savefig(FIG / "fig_e1_severity_pr_calibration.png"); plt.show()

In [ ]:
import shap
Xs = X["test"].sample(5000, random_state=cfg["seed"])
sv = shap.TreeExplainer(gbt).shap_values(Xs)
imp = pd.Series(np.abs(sv).mean(0), index=Xs.columns).sort_values(ascending=False)
imp.to_csv(TAB / "e1_gbt_shap_importance.csv")
fig, ax = plt.subplots(figsize=(6, 4))
imp.head(15).iloc[::-1].plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("mean |SHAP value| (log-odds)"); ax.set_title("Gradient-boosted trees: feature importance")
fig.savefig(FIG / "fig_e1_shap.png"); plt.show()

## 5. Hypothesis H1

In [ ]:
h1 = pd.DataFrame({
    "estimate": [irr.loc["rain", "estimate"], irr.loc["snow", "estimate"], orr.loc["rain", "estimate"], orr.loc["snow", "estimate"]],
    "ci_low": [irr.loc["rain", "ci_low"], irr.loc["snow", "ci_low"], orr.loc["rain", "ci_low"], orr.loc["snow", "ci_low"]],
    "ci_high": [irr.loc["rain", "ci_high"], irr.loc["snow", "ci_high"], orr.loc["rain", "ci_high"], orr.loc["snow", "ci_high"]]},
    index=["IRR rain (frequency)", "IRR snow (frequency)", "OR rain (severity)", "OR snow (severity)"])
h1["verdict (expected > 1)"] = [M.hypothesis_verdict(r.estimate, r.ci_low, r.ci_high) for r in h1.itertuples()]
h1.to_csv(TAB / "h1.csv")
h1.round(3)